# 03. Feature Engineering and Analytical Dataset

## Objective
This notebook integrates cleaned operational and master project data to create analytical features for system performance monitoring and predictive maintenance.

## Inputs
- Cleaned consolidated dataset (`consolidated_silver.csv`)
- Cleaned master dataset (`master_silver.csv`)

## Output
- Gold analytical dataset with performance indicators and maintenance flags.

In [233]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading

This section loads the cleaned datasets from the silver layer and prepares them for integration.

In [234]:
consolidated_path = "../data/silver/consolidated_silver.csv"
master_path = "../data/silver/master_silver.csv"

df_cons = pd.read_csv(consolidated_path)
df_master = pd.read_csv(master_path)

df_cons.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),energia especifica (kwh/kwp),consumo (kwh),autoconsumo (kwh),potencia maxima (kw),co2 evitado (t),anio,mes_num,month_year,generacion_mensual_kwh
0,EVT-2021-0001,2025-01-01,151.34,0.00,0.0,0.0,25.712,0.0,0.0,0.000,0.0136,2025,1,2025-01,5236.83
1,EVT-2021-0002,2025-01-01,0.00,0.00,0.0,0.0,0.000,0.0,0.0,0.000,0.0000,2025,1,2025-01,2534.99
2,EVT-2021-0003,2025-01-01,77.80,0.00,0.0,0.0,27.184,0.0,0.0,0.000,0.0070,2025,1,2025-01,195.74
3,EVT-2022-0004,2025-01-01,156.80,0.00,0.0,0.0,25.712,0.0,0.0,0.000,0.0130,2025,1,2025-01,5577.70
4,EVT-2022-0005,2025-01-01,6.51,9086.07,0.0,0.0,2.893,0.0,0.0,2.009,0.0030,2025,1,2025-01,285.93


In [235]:
df_master.head()

,id_proyecto,nombre_proyecto,razon_social_cliente,contacto_cliente,contrato,plazo_anos,entrega_equipos,tarifa_o_descuento,descuento_porcentaje,tarifa_inicial_cop_kwh,...,generacion_mensual_de_la_propuesta,ahorro_mensual_cotizacion,hsp,pr_pvsyst,fpo_original,fecha_ultimo_mantenimiento_original,entrega_equipos_flag,fpo_estado,mantenimiento_estado,estado_operativo
0,EVT-2021-0001,La Merced Ipiales,INELCO .S.A.S.,3185309470,EPC,NaN,NaN,NaN,NaN,NaN,...,5509.0,NaN,3.90,0.8,14/06/2022,7/07/2024,NaN,fecha_valida,fecha_valida,en_marcha
1,EVT-2021-0002,Seguridad del Sur,SEGURIDAD DEL SUR LIMITADA,3046581766,EPC,NaN,NaN,NaN,NaN,NaN,...,4000.0,NaN,3.80,0.8,6/11/2021,23/03/2023,NaN,fecha_valida,fecha_valida,en_marcha
2,EVT-2021-0003,Dispropan Pasto,INVEESE S.A.S,3137501606,EPC,NaN,NaN,Tarifa,NaN,713.0,...,3200.0,NaN,4.10,0.8,7/03/2022,24/06/2024,NaN,fecha_valida,fecha_valida,en_marcha
3,EVT-2022-0004,La Merced Aurora,INVEESE S.A.S,3183659405,EPC,NaN,NaN,NaN,NaN,NaN,...,6314.0,NaN,4.09,0.8,29/10/2022,17/06/2024,NaN,fecha_valida,fecha_valida,en_marcha
4,EVT-2022-0005,Casa Woodcock,HENRY WOODCOCK DELGADO,NaN,EPC,NaN,NaN,NaN,NaN,NaN,...,216.0,NaN,4.10,0.8,13/03/2022,18/10/2023,NaN,fecha_valida,fecha_valida,en_marcha


In [236]:
df_cons.shape

(16932, 15)

In [237]:
df_master.shape

(105, 38)

In [238]:
df_cons.columns.tolist()

['id_proyecto',
 'fecha',
 'rendimiento fv (kwh)',
 'energia acumulada (kwh)',
 'energia exportada (kwh)',
 'importacion (kwh)',
 'energia especifica (kwh/kwp)',
 'consumo (kwh)',
 'autoconsumo (kwh)',
 'potencia maxima (kw)',
 'co2 evitado (t)',
 'anio',
 'mes_num',
 'month_year',
 'generacion_mensual_kwh']

In [239]:
df_master.columns.tolist()

['id_proyecto',
 'nombre_proyecto',
 'razon_social_cliente',
 'contacto_cliente',
 'contrato',
 'plazo_anos',
 'entrega_equipos',
 'tarifa_o_descuento',
 'descuento_porcentaje',
 'tarifa_inicial_cop_kwh',
 'tipo',
 'tipo_2',
 'potencia_instalada_kw',
 'cantidad_de_paneles',
 'modelo_de_paneles',
 'potencia_panel',
 'cantidad_inversor',
 'modelo_de_inversores',
 'potencia_inversores_kw',
 'area_m2',
 'ciudad',
 'departamento',
 'porcentaje_ahorro_proyectado_energia',
 'fecha_firma_de_contrato',
 'fpo',
 'fecha_de_energizacion',
 'fecha_ultimo_mantenimiento',
 'consumo_anterior_al_sistema_kwh',
 'generacion_mensual_de_la_propuesta',
 'ahorro_mensual_cotizacion',
 'hsp',
 'pr_pvsyst',
 'fpo_original',
 'fecha_ultimo_mantenimiento_original',
 'entrega_equipos_flag',
 'fpo_estado',
 'mantenimiento_estado',
 'estado_operativo']

In [240]:
df_cons["fecha"] = pd.to_datetime(df_cons["fecha"], errors="coerce")

df_master["fpo"] = pd.to_datetime(df_master["fpo"], errors="coerce")
df_master["fecha_de_energizacion"] = pd.to_datetime(df_master["fecha_de_energizacion"], errors="coerce")
df_master["fecha_ultimo_mantenimiento"] = pd.to_datetime(df_master["fecha_ultimo_mantenimiento"], errors="coerce")

## Data Integration

The cleaned operational dataset is merged with the master project dataset using `id_proyecto` as the primary key.

In [241]:
df_master["id_proyecto"].duplicated().sum()

np.int64(0)

In [242]:
df_gold = df_cons.merge(df_master, on="id_proyecto", how="left")

In [243]:
df_gold.shape

(16932, 52)

In [244]:
df_gold.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia acumulada (kwh),energia exportada (kwh),importacion (kwh),energia especifica (kwh/kwp),consumo (kwh),autoconsumo (kwh),potencia maxima (kw),...,generacion_mensual_de_la_propuesta,ahorro_mensual_cotizacion,hsp,pr_pvsyst,fpo_original,fecha_ultimo_mantenimiento_original,entrega_equipos_flag,fpo_estado,mantenimiento_estado,estado_operativo
0,EVT-2021-0001,2025-01-01,151.34,0.00,0.0,0.0,25.712,0.0,0.0,0.000,...,5509.0,NaN,3.90,0.8,14/06/2022,7/07/2024,NaN,fecha_valida,fecha_valida,en_marcha
1,EVT-2021-0002,2025-01-01,0.00,0.00,0.0,0.0,0.000,0.0,0.0,0.000,...,4000.0,NaN,3.80,0.8,6/11/2021,23/03/2023,NaN,fecha_valida,fecha_valida,en_marcha
2,EVT-2021-0003,2025-01-01,77.80,0.00,0.0,0.0,27.184,0.0,0.0,0.000,...,3200.0,NaN,4.10,0.8,7/03/2022,24/06/2024,NaN,fecha_valida,fecha_valida,en_marcha
3,EVT-2022-0004,2025-01-01,156.80,0.00,0.0,0.0,25.712,0.0,0.0,0.000,...,6314.0,NaN,4.09,0.8,29/10/2022,17/06/2024,NaN,fecha_valida,fecha_valida,en_marcha
4,EVT-2022-0005,2025-01-01,6.51,9086.07,0.0,0.0,2.893,0.0,0.0,2.009,...,216.0,NaN,4.10,0.8,13/03/2022,18/10/2023,NaN,fecha_valida,fecha_valida,en_marcha


In [245]:
df_gold.isnull().sum().sort_values(ascending=False).head(20)

tarifa_inicial_cop_kwh                  14148
entrega_equipos_flag                    13352
ahorro_mensual_cotizacion               12806
descuento_porcentaje                    12752
fecha_ultimo_mantenimiento              10839
fecha_de_energizacion                   10664
tarifa_o_descuento                      10020
entrega_equipos                         10018
plazo_anos                              10018
fecha_ultimo_mantenimiento_original      8797
contacto_cliente                         3168
energia especifica (kwh/kwp)             2911
potencia maxima (kw)                     1621
fpo                                      1451
porcentaje_ahorro_proyectado_energia     1070
consumo_anterior_al_sistema_kwh           562
potencia_instalada_kw                     533
fpo_original                              438
area_m2                                   229
generacion_mensual_de_la_propuesta         79
dtype: int64

## Feature Engineering and Performance Metrics

In this section, several analytical features were created to evaluate the operational behavior of photovoltaic systems and identify potential performance anomalies.

In [246]:
df_gold["energia_teorica_kwh"] = df_gold["potencia_instalada_kw"] * df_gold["hsp"]

Estimated as the product of installed capacity and average solar resource. This provides a baseline reference for expected system generation.
  

In [247]:
df_gold["pr_real"] = np.where(
    df_gold["energia_teorica_kwh"] > 0,
    df_gold["rendimiento fv (kwh)"] / df_gold["energia_teorica_kwh"],
    np.nan
)

This provides a baseline reference for expected system generation.

In [248]:
df_gold["desviacion_pr"] = df_gold["pr_real"] - df_gold["pr_pvsyst"]

Measures how the system is performing relative to its theoretical potential. Negative values indicate underperformance.

In [249]:
df_gold["dias_desde_ultimo_mantenimiento"] = (
    df_gold["fecha"] - df_gold["fecha_ultimo_mantenimiento"]
).dt.days

Measures elapsed time since the last recorded maintenance event

In [250]:
df_gold["dias_desde_fpo"] = (df_gold["fecha"] - df_gold["fpo"]).dt.days

Measures elapsed time since the last recorded maintenance event

In [251]:
df_gold = df_gold.sort_values(["id_proyecto", "fecha"])

df_gold["variacion_diaria_generacion"] = (
    df_gold.groupby("id_proyecto")["rendimiento fv (kwh)"].pct_change()
)

Captures relative changes in daily generation per project to detect abrupt drops.

In [252]:
df_gold["inactividad"] = df_gold["rendimiento fv (kwh)"] <= 0

Captures relative changes in daily generation per project to detect abrupt drops.

In [253]:
def clasificar_alerta(row):
    
    
    if pd.notnull(row["fpo_estado"]) and row["fpo_estado"] in ["limitado", "en_curso"]:
        return "revision_estado_proyecto"

    if row["inactividad"]:
        return "alerta_critica"
    
    if pd.isnull(row["pr_real"]) or pd.isnull(row["pr_pvsyst"]):
        return "sin_informacion"
    
    if row["pr_real"] >= 0.90 * row["pr_pvsyst"]:
        return "operacion_esperada"
    
    elif row["pr_real"] >= 0.75 * row["pr_pvsyst"]:
        return "posible_degradacion"
    
    else:
        return "bajo_desempeno"

In [254]:
df_gold["clasificacion_alerta"] = df_gold.apply(clasificar_alerta, axis=1)

In [255]:
df_gold["clasificacion_alerta"].value_counts(dropna=False)

clasificacion_alerta
operacion_esperada          7992
bajo_desempeno              3819
posible_degradacion         2928
alerta_critica              1594
sin_informacion              434
revision_estado_proyecto     165
Name: count, dtype: int64

Categorizes system behavior into states such as:
  - expected operation
  - possible degradation
  - low performance
  - critical alert

  This classification is based on relative performance thresholds and system activity.

## Potential Soiling Alert

A potential soiling alert was created to identify projects showing sustained moderate underperformance without complete inactivity.

This rule does not confirm soiling as the root cause, but highlights patterns compatible with gradual performance degradation that may warrant inspection.

In [256]:
df_gold["ratio_desempeno"] = np.where(
    (df_gold["pr_pvsyst"] > 0) & (df_gold["pr_real"].notna()),
    df_gold["pr_real"] / df_gold["pr_pvsyst"],
    np.nan
)

Normalizes real performance against expected performance. Normalizes real performance against expected performance

In [257]:
df_gold["ratio_desempeno_7d"] = (
    df_gold.groupby("id_proyecto")["ratio_desempeno"]
    .transform(lambda x: x.rolling(7, min_periods=4).mean())
)

A 7-day moving average used to smooth short-term variability and reduce noise from external factors such as weather.

In [258]:
df_gold["alerta_posible_ensuciamiento"] = np.where(
    (df_gold["ratio_desempeno_7d"] < 0.90) &
    (df_gold["ratio_desempeno_7d"] >= 0.75) &
    (~df_gold["inactividad"]) &
    (df_gold["clasificacion_alerta"] == "posible_degradacion"),
    True,
    False
)

In [259]:
df_gold[["ratio_desempeno", "ratio_desempeno_7d"]].head(10)

,ratio_desempeno,ratio_desempeno_7d
0,0.824939,NaN
69,0.644296,NaN
138,0.842709,NaN
207,0.663320,0.743816
261,0.437925,0.682638
330,0.803789,0.702830
399,0.759964,0.710992
471,1.048044,0.742864
540,0.955979,0.787390
625,0.922837,0.798837


In [260]:
df_gold["alerta_posible_ensuciamiento"].value_counts(dropna=False)

alerta_posible_ensuciamiento
False    15864
True      1068
Name: count, dtype: int64

In [261]:
df_gold.loc[df_gold["alerta_posible_ensuciamiento"] == True, [
    "id_proyecto",
    "fecha",
    "pr_real",
    "pr_pvsyst",
    "ratio_desempeno",
    "ratio_desempeno_7d",
    "clasificacion_alerta"
]].head(20)

,id_proyecto,fecha,pr_real,pr_pvsyst,ratio_desempeno,ratio_desempeno_7d,clasificacion_alerta
4283,EVT-2021-0001,2025-03-01,0.644209,0.8,0.805261,0.784088,posible_degradacion
4358,EVT-2021-0001,2025-03-02,0.655765,0.8,0.819706,0.806507,posible_degradacion
4433,EVT-2021-0001,2025-03-03,0.604526,0.8,0.755658,0.783146,posible_degradacion
4506,EVT-2021-0001,2025-03-04,0.604526,0.8,0.755658,0.778933,posible_degradacion
5180,EVT-2021-0001,2025-03-13,0.602608,0.8,0.753260,0.779517,posible_degradacion
14946,EVT-2021-0001,2025-07-10,0.689997,0.8,0.862496,0.816591,posible_degradacion
1957,EVT-2021-0002,2025-01-29,0.662609,0.8,0.828261,0.801617,posible_degradacion
4284,EVT-2021-0002,2025-03-01,0.610483,0.8,0.763103,0.820063,posible_degradacion
4434,EVT-2021-0002,2025-03-03,0.654684,0.8,0.818355,0.781829,posible_degradacion
4951,EVT-2021-0002,2025-03-10,0.709474,0.8,0.886843,0.826809,posible_degradacion


In [262]:
gold_columns = [
    "id_proyecto",
    "fecha",
    "rendimiento fv (kwh)",
    "energia especifica (kwh/kwp)",
    "potencia maxima (kw)",
    "generacion_mensual_kwh",
    "potencia_instalada_kw",
    "hsp",
    "pr_pvsyst",
    "energia_teorica_kwh",
    "pr_real",
    "desviacion_pr",
    "dias_desde_ultimo_mantenimiento",
    "dias_desde_fpo",
    "variacion_diaria_generacion",
    "inactividad",
    "fpo_estado",
    "mantenimiento_estado",
    "entrega_equipos_flag",
    "clasificacion_alerta",
    "ratio_desempeno",
    "ratio_desempeno_7d",
    "alerta_posible_ensuciamiento"
]

df_gold_final = df_gold[gold_columns].copy()
df_gold_final.head()

,id_proyecto,fecha,rendimiento fv (kwh),energia especifica (kwh/kwp),potencia maxima (kw),generacion_mensual_kwh,potencia_instalada_kw,hsp,pr_pvsyst,energia_teorica_kwh,...,dias_desde_fpo,variacion_diaria_generacion,inactividad,fpo_estado,mantenimiento_estado,entrega_equipos_flag,clasificacion_alerta,ratio_desempeno,ratio_desempeno_7d,alerta_posible_ensuciamiento
0,EVT-2021-0001,2025-01-01,151.34,25.712,0.0,5236.83,58.8,3.9,0.8,229.32,...,932.0,NaN,False,fecha_valida,fecha_valida,NaN,posible_degradacion,0.824939,NaN,False
69,EVT-2021-0001,2025-01-02,118.20,20.082,0.0,5236.83,58.8,3.9,0.8,229.32,...,933.0,-0.218977,False,fecha_valida,fecha_valida,NaN,bajo_desempeno,0.644296,NaN,False
138,EVT-2021-0001,2025-01-03,154.60,26.266,0.0,5236.83,58.8,3.9,0.8,229.32,...,934.0,0.307953,False,fecha_valida,fecha_valida,NaN,posible_degradacion,0.842709,NaN,False
207,EVT-2021-0001,2025-01-04,121.69,20.674,0.0,5236.83,58.8,3.9,0.8,229.32,...,935.0,-0.212872,False,fecha_valida,fecha_valida,NaN,bajo_desempeno,0.663320,0.743816,False
261,EVT-2021-0001,2025-01-05,80.34,13.649,0.0,5236.83,58.8,3.9,0.8,229.32,...,936.0,-0.339798,False,fecha_valida,fecha_valida,NaN,bajo_desempeno,0.437925,0.682638,False


In [263]:
df_gold_final.to_csv("../data/gold/gold_analytical_dataset.csv", index=False)

## Final Output

The final gold dataset integrates operational and project-level data into a unified analytical layer.

It includes performance indicators, alert classifications, and a potential soiling signal designed to support anomaly detection and maintenance prioritization in photovoltaic systems.